# 09 - Motion Metrics for Bidirectional ConvLSTM Input Sequences

This notebook computes motion information for the exact 25-frame input windows used by the trained bidirectional ConvLSTM U-Net and by `08_seg_gradcam.ipynb`.

For each official EchoNet-Dynamic test sample, it saves numerical motion arrays for the 24 consecutive transitions between selected sequence frames:

- dense Farneback optical flow, shaped `[24, H, W, 2]`
- optical-flow magnitude, shaped `[24, H, W]`
- absolute frame difference, shaped `[24, H, W]`

The optical-flow direction convention is: `optical_flow[t]` maps coordinates from selected sequence frame `t` toward selected sequence frame `t+1`. The vector components are `(dx, dy)` in image coordinates, where `dx` is horizontal displacement and `dy` is vertical displacement.

This notebook does not train the model, load the segmentation model, regenerate Seg-Grad-CAM heatmaps, or compute motion from visualization PNGs.


## Kaggle Setup and Outputs

The current implementation uses OpenCV Farneback optical flow, which is CPU-based. A Kaggle T4 x2 accelerator is fine, but the GPUs are not used by this Farneback pipeline. The notebook still reports CUDA/GPU visibility for reproducibility.

Expected output layout:

```text
/kaggle/working/motion_metrics/
  samples/
    <sample_id>_<hash>.npz
  quality_control/
  motion_manifest.csv
  failed_motion_samples.csv
  summary.json
```

Before ending a Kaggle session, download `/kaggle/working/motion_metrics` or save it as a Kaggle Dataset. `/kaggle/working` is not permanent after the session is discarded.


In [ ]:
# Optional. Kaggle usually already has these packages, but this is safe to run if needed.
%pip install -q opencv-python-headless pandas matplotlib tqdm


## Configuration and Paths

Update the paths below for your Kaggle datasets. Use `FLAT_SOURCE_LAYOUT=True` if your source-code dataset contains files directly, such as `dataset.py`, `utils.py`, etc. Use `False` if it contains a nested `src/` package.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib
import json
import math
import os
import random
import shutil
import sys
import tempfile
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# -----------------------------
# Path configuration
# -----------------------------
PROJECT_ROOT = Path("/kaggle/working/Echonet_temporal_XAI")
RAW_DIR = Path("/kaggle/input/echonet-dynamic-raw/EchoNet-Dynamic")
PROCESSED_DIR = Path("/kaggle/input/echonet-dynamic-processed-masks")
SEG_GRADCAM_OUTPUT_ROOT = Path("/kaggle/working/seg_gradcam_outputs")
OUTPUT_ROOT = Path("/kaggle/working/motion_metrics")

# Set True if PROJECT_ROOT contains dataset.py/utils.py directly rather than src/dataset.py.
FLAT_SOURCE_LAYOUT = False

# Same input-window configuration used by notebook 08 and the bidirectional model.
NUM_FRAMES_BEFORE = 12
NUM_FRAMES_AFTER = 12
TEMPORAL_STRIDE = 2
TARGET_POSITION = 12
SEQUENCE_LENGTH = 25
IMAGE_SIZE = (112, 112)
SEED = 42

# Run controls.
RUN_MODE = "smoke"  # "smoke" or "full"
SMOKE_MAX_SAMPLES = 3
FULL_MAX_SAMPLES = None
OVERWRITE_EXISTING = False
SAVE_INPUT_FRAMES = True
FAIL_FAST_SMOKE = True

# Quality-control visualization controls.
QC_SAMPLE_COUNT = 12
QC_SEED = 42
QC_MAX_TRANSITIONS_PER_SAMPLE = 8
QC_ARROW_STEP = 12
QC_FLOW_WARNING_MAGNITUDE_PX = 25.0

# Output layout.
SAMPLES_DIR = OUTPUT_ROOT / "samples"
QC_DIR = OUTPUT_ROOT / "quality_control"
MANIFEST_PATH = OUTPUT_ROOT / "motion_manifest.csv"
FAILED_PATH = OUTPUT_ROOT / "failed_motion_samples.csv"
SUMMARY_PATH = OUTPUT_ROOT / "summary.json"
QC_MANIFEST_PATH = QC_DIR / "qc_manifest.csv"

for directory in [OUTPUT_ROOT, SAMPLES_DIR, QC_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_DIR: {RAW_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")
print(f"SEG_GRADCAM_OUTPUT_ROOT: {SEG_GRADCAM_OUTPUT_ROOT}")
print(f"OUTPUT_ROOT: {OUTPUT_ROOT}")
print(f"RUN_MODE: {RUN_MODE}")


## Environment Report

Farneback optical flow runs on CPU. CUDA is reported only so the Kaggle runtime configuration is documented. The notebook does not use `DataParallel`, does not load the trained segmentation model, and does not allocate model tensors on GPU.


In [ ]:
print(f"Python executable: {sys.executable}")
print(f"OpenCV version: {cv2.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Visible GPU count: {torch.cuda.device_count()}")
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {props.name}, total={props.total_memory / 1024**3:.2f} GiB")
else:
    print("Visible GPU count: 0")
print("Pipeline compute device: CPU for OpenCV Farneback optical flow")


## Imports From Project Code

The motion notebook reuses the same split and dataset code used by notebook 08. This keeps sample IDs, test-set ordering, target-frame definitions, preprocessing geometry, frame clamping, and frame indices synchronized with Grad-CAM generation.


In [ ]:
def prepare_import_root(project_root: Path, flat_source_layout: bool) -> Path:
    if not flat_source_layout:
        if not (project_root / "src").exists():
            raise FileNotFoundError(
                f"Expected src/ under PROJECT_ROOT={project_root}. "
                "Set FLAT_SOURCE_LAYOUT=True if your source dataset contains .py files directly."
            )
        return project_root

    staged_root = Path("/kaggle/working/_motion_project_src") if Path("/kaggle/working").exists() else OUTPUT_ROOT / "_motion_project_src"
    staged_src = staged_root / "src"
    staged_src.mkdir(parents=True, exist_ok=True)
    for py_file in project_root.glob("*.py"):
        shutil.copy2(py_file, staged_src / py_file.name)
    (staged_src / "__init__.py").touch()
    if not (staged_src / "dataset.py").exists():
        raise FileNotFoundError(f"Flat source folder did not contain dataset.py: {project_root}")
    return staged_root


IMPORT_ROOT = prepare_import_root(PROJECT_ROOT, FLAT_SOURCE_LAYOUT)
if str(IMPORT_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPORT_ROOT))

dataset_module = importlib.import_module("src.dataset")
utils_module = importlib.import_module("src.utils")

EchoNetTemporalDataset = dataset_module.EchoNetTemporalDataset
load_temporal_metadata = dataset_module.load_temporal_metadata
split_by_echonet_filelist = dataset_module.split_by_echonet_filelist
load_echonet_tables = utils_module.load_echonet_tables

print(f"Imported project modules from: {IMPORT_ROOT}")


## Dataset and Exact-Frame Verification

The dataset is built with `augment=False`, `num_frames_before=12`, `num_frames_after=12`, and `temporal_stride=2`. Its returned `frame_indices` are the original EchoNet video-frame indices after the same clamping policy used for model inference and Grad-CAM generation.


In [ ]:
def infer_target_type(samples_for_video: list[dict[str, Any]], sample: dict[str, Any]) -> str:
    # EchoNet processed metadata has two labeled frames per video. In this project table, the smaller index is treated as ED.
    frame_indices = sorted(int(item["frame_idx"]) for item in samples_for_video)
    target_frame = int(sample["frame_idx"])
    if len(frame_indices) >= 2:
        if target_frame == frame_indices[0]:
            return "ED"
        if target_frame == frame_indices[-1]:
            return "ES"
    return "unknown"


def load_official_test_samples() -> list[dict[str, Any]]:
    metadata_path = PROCESSED_DIR / "metadata.csv"
    if not metadata_path.exists():
        raise FileNotFoundError(f"metadata.csv not found: {metadata_path}")
    if not (RAW_DIR / "FileList.csv").exists():
        raise FileNotFoundError(f"FileList.csv not found: {RAW_DIR / 'FileList.csv'}")
    if not (RAW_DIR / "Videos").exists():
        raise FileNotFoundError(f"Videos directory not found: {RAW_DIR / 'Videos'}")

    samples = load_temporal_metadata(metadata_path)
    file_list, _ = load_echonet_tables(RAW_DIR)
    _, _, test_samples = split_by_echonet_filelist(samples, file_list)

    by_video: dict[str, list[dict[str, Any]]] = {}
    for sample in samples:
        by_video.setdefault(str(sample["video_id"]), []).append(sample)

    enriched = []
    for sample in test_samples:
        item = dict(sample)
        item["target_type"] = infer_target_type(by_video.get(str(sample["video_id"]), []), item)
        enriched.append(item)
    return enriched


all_test_samples = load_official_test_samples()
active_samples = all_test_samples[:SMOKE_MAX_SAMPLES] if RUN_MODE == "smoke" else all_test_samples[:FULL_MAX_SAMPLES]

motion_dataset = EchoNetTemporalDataset(
    active_samples,
    videos_dir=RAW_DIR / "Videos",
    num_frames_before=NUM_FRAMES_BEFORE,
    num_frames_after=NUM_FRAMES_AFTER,
    temporal_stride=TEMPORAL_STRIDE,
    image_size=IMAGE_SIZE,
    augment=False,
)

motion_loader = DataLoader(
    motion_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

print(f"Official test samples: {len(all_test_samples):,}")
print(f"Active {RUN_MODE} samples: {len(active_samples):,}")
print(pd.Series([sample["target_type"] for sample in active_samples]).value_counts(dropna=False))

sample0 = motion_dataset[0]
assert sample0["sequence"].shape == (SEQUENCE_LENGTH, 1, *IMAGE_SIZE)
assert int(sample0["target_idx"]) == TARGET_POSITION
assert int(sample0["frame_indices"][TARGET_POSITION]) == int(sample0["frame_idx"])
assert bool(motion_dataset.augment) is False
print(f"Example sample id: {sample0['id']}")
print(f"Sequence shape: {tuple(sample0['sequence'].shape)}")
print(f"Target position: {sample0['target_idx']}")
print(f"Selected frame indices: {sample0['frame_indices'].tolist()}")


## Optical-Flow Implementation

The initial implementation uses OpenCV Farneback dense optical flow behind a small estimator interface:

```python
flow_estimator.compute(frame_t, frame_t1) -> flow
```

Frames are already resized by `EchoNetTemporalDataset` to `IMAGE_SIZE` and normalized to `[0, 1]`. Farneback receives `float32` grayscale frames in that same resized coordinate system. Therefore saved flow vectors are displacements in the model/Grad-CAM spatial coordinate system (`112 x 112` by default), and no post-hoc displacement scaling is applied.

Farneback parameters are explicitly configured below. Future estimators such as RAFT can implement the same `compute(frame_t, frame_t1)` method without changing the dataset or saving pipeline.


In [ ]:
@dataclass(frozen=True)
class FarnebackConfig:
    pyr_scale: float = 0.5
    levels: int = 3
    winsize: int = 15
    iterations: int = 3
    poly_n: int = 5
    poly_sigma: float = 1.2
    flags: int = 0


class DenseFlowEstimator:
    method_name: str = "abstract"

    def compute(self, frame_t: np.ndarray, frame_t1: np.ndarray) -> np.ndarray:
        raise NotImplementedError


class FarnebackFlowEstimator(DenseFlowEstimator):
    method_name = "opencv_farneback"

    def __init__(self, config: FarnebackConfig) -> None:
        self.config = config

    def compute(self, frame_t: np.ndarray, frame_t1: np.ndarray) -> np.ndarray:
        frame_t = np.asarray(frame_t, dtype=np.float32)
        frame_t1 = np.asarray(frame_t1, dtype=np.float32)
        if frame_t.shape != frame_t1.shape:
            raise ValueError(f"Frame shapes differ: {frame_t.shape} vs {frame_t1.shape}")
        if frame_t.ndim != 2:
            raise ValueError(f"Expected 2D grayscale frames, got shape {frame_t.shape}")
        flow = cv2.calcOpticalFlowFarneback(
            frame_t,
            frame_t1,
            None,
            pyr_scale=self.config.pyr_scale,
            levels=self.config.levels,
            winsize=self.config.winsize,
            iterations=self.config.iterations,
            poly_n=self.config.poly_n,
            poly_sigma=self.config.poly_sigma,
            flags=self.config.flags,
        )
        return flow.astype(np.float32)


FLOW_CONFIG = FarnebackConfig()
flow_estimator = FarnebackFlowEstimator(FLOW_CONFIG)
print(FLOW_CONFIG)


## Motion and Frame-Difference Computation

Frame differences are computed as `abs(frame[t+1] - frame[t])` on the same `[0, 1]` float32 grayscale frames supplied to the model. No random augmentation is active.


In [ ]:
def compute_motion_arrays(frames: np.ndarray, estimator: DenseFlowEstimator) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    frames = np.asarray(frames, dtype=np.float32)
    if frames.shape != (SEQUENCE_LENGTH, *IMAGE_SIZE):
        raise ValueError(f"Expected frames shape {(SEQUENCE_LENGTH, *IMAGE_SIZE)}, got {frames.shape}")
    flows = []
    magnitudes = []
    differences = []
    for transition_idx in range(SEQUENCE_LENGTH - 1):
        frame_t = frames[transition_idx]
        frame_t1 = frames[transition_idx + 1]
        flow = estimator.compute(frame_t, frame_t1)
        magnitude = np.sqrt(np.square(flow[..., 0]) + np.square(flow[..., 1])).astype(np.float32)
        difference = np.abs(frame_t1 - frame_t).astype(np.float32)
        flows.append(flow)
        magnitudes.append(magnitude)
        differences.append(difference)
    optical_flow = np.stack(flows, axis=0).astype(np.float32)
    flow_magnitude = np.stack(magnitudes, axis=0).astype(np.float32)
    frame_difference = np.stack(differences, axis=0).astype(np.float32)
    return optical_flow, flow_magnitude, frame_difference


def validate_motion_arrays(
    frames: np.ndarray,
    optical_flow: np.ndarray,
    flow_magnitude: np.ndarray,
    frame_difference: np.ndarray,
) -> None:
    assert frames.shape == (SEQUENCE_LENGTH, *IMAGE_SIZE)
    assert optical_flow.shape == (SEQUENCE_LENGTH - 1, *IMAGE_SIZE, 2)
    assert flow_magnitude.shape == (SEQUENCE_LENGTH - 1, *IMAGE_SIZE)
    assert frame_difference.shape == (SEQUENCE_LENGTH - 1, *IMAGE_SIZE)
    for name, array in [
        ("frames", frames),
        ("optical_flow", optical_flow),
        ("flow_magnitude", flow_magnitude),
        ("frame_difference", frame_difference),
    ]:
        if not np.isfinite(array).all():
            raise ValueError(f"{name} contains NaN or infinite values")


## Saving, Resumption, and Integrity Checks

Each sample is saved atomically to one compressed `.npz` file. Resume logic validates existing files before skipping them. Files are rejected if they are missing required keys, have wrong shapes, contain object arrays, or contain NaN/Inf values.


In [ ]:
def sanitize_id(value: str) -> str:
    return "".join(char if char.isalnum() or char in {"-", "_"} else "_" for char in str(value))


def sample_output_path(sample_id: str) -> Path:
    safe = sanitize_id(sample_id)
    digest = hashlib.sha1(sample_id.encode("utf-8")).hexdigest()[:8]
    return SAMPLES_DIR / f"{safe}_{digest}.npz"


def atomic_save_npz(path: Path, **arrays: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=path.parent, suffix=".npz", delete=False) as tmp:
        tmp_path = Path(tmp.name)
    try:
        np.savez_compressed(tmp_path, **arrays)
        tmp_path.replace(path)
    finally:
        if tmp_path.exists():
            tmp_path.unlink()


def append_manifest_row(path: Path, row: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    row_df = pd.DataFrame([row])
    if path.exists() and path.stat().st_size > 0:
        row_df.to_csv(path, mode="a", index=False, header=False)
    else:
        row_df.to_csv(path, index=False)


def read_manifest(path: Path) -> pd.DataFrame:
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def validate_saved_npz(path: Path, expected_hw: tuple[int, int] = IMAGE_SIZE) -> bool:
    required = {
        "sample_id",
        "video_id",
        "target_type",
        "target_frame_index",
        "frame_indices",
        "sequence_positions",
        "optical_flow",
        "flow_magnitude",
        "frame_difference",
        "stride",
        "target_position",
        "padding_mask",
        "preprocessing_metadata",
        "flow_direction_convention",
    }
    if SAVE_INPUT_FRAMES:
        required.add("frames")
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        with np.load(path, allow_pickle=False) as data:
            if any(key not in data for key in required):
                return False
            for key in data.files:
                array = data[key]
                if array.dtype == object:
                    return False
            h, w = expected_hw
            if data["frame_indices"].shape != (SEQUENCE_LENGTH,):
                return False
            if data["sequence_positions"].shape != (SEQUENCE_LENGTH,):
                return False
            if data["optical_flow"].shape != (SEQUENCE_LENGTH - 1, h, w, 2):
                return False
            if data["flow_magnitude"].shape != (SEQUENCE_LENGTH - 1, h, w):
                return False
            if data["frame_difference"].shape != (SEQUENCE_LENGTH - 1, h, w):
                return False
            if SAVE_INPUT_FRAMES and data["frames"].shape != (SEQUENCE_LENGTH, h, w):
                return False
            for key in ["optical_flow", "flow_magnitude", "frame_difference"]:
                if not np.isfinite(data[key]).all():
                    return False
            if SAVE_INPUT_FRAMES and not np.isfinite(data["frames"]).all():
                return False
        return True
    except Exception:
        return False


def preprocessing_metadata_json() -> np.ndarray:
    metadata = {
        "source": "src.dataset.EchoNetTemporalDataset",
        "image_size": list(IMAGE_SIZE),
        "frame_scale": "float32 grayscale normalized to [0, 1]",
        "resize_interpolation": "cv2.INTER_AREA for frames",
        "augmentation": False,
        "coordinate_system": "same resized HxW image coordinate system as model input and Grad-CAM heatmaps",
        "flow_displacement_units": "pixels in resized model/Grad-CAM coordinate system",
        "flow_resizing": "none after dataset preprocessing; no displacement scaling applied",
    }
    return np.asarray(json.dumps(metadata), dtype="U2048")


def flow_config_json() -> np.ndarray:
    return np.asarray(json.dumps(FLOW_CONFIG.__dict__), dtype="U1024")


## Per-Sample Processing

The saved `padding_mask` marks sequence positions where the requested frame index was clamped to a video boundary. Repeated frame indices are also counted in the manifest because clamped boundary windows can produce duplicated frames.


In [ ]:
def batch_scalar(batch: dict[str, Any], key: str) -> Any:
    value = batch[key]
    if isinstance(value, torch.Tensor):
        selected = value[0]
        if selected.ndim == 0:
            return selected.item()
        return selected.detach().cpu().numpy()
    if isinstance(value, (list, tuple)):
        return value[0]
    return value


def sample_meta(dataset_index: int, batch: dict[str, Any]) -> dict[str, Any]:
    sample = active_samples[dataset_index]
    return {
        "sample_id": str(batch_scalar(batch, "id")),
        "video_id": str(batch_scalar(batch, "video_id")),
        "target_type": str(sample.get("target_type", "unknown")),
        "target_frame_index": int(batch_scalar(batch, "frame_idx")),
    }


def requested_unclamped_indices(target_frame_index: int) -> np.ndarray:
    offsets = np.asarray([
        offset * TEMPORAL_STRIDE for offset in range(-NUM_FRAMES_BEFORE, NUM_FRAMES_AFTER + 1)
    ], dtype=np.int64)
    return target_frame_index + offsets


def process_one_sample(dataset_index: int, batch: dict[str, Any]) -> dict[str, Any]:
    meta = sample_meta(dataset_index, batch)
    sample_id = meta["sample_id"]
    output_path = sample_output_path(sample_id)

    if not OVERWRITE_EXISTING and validate_saved_npz(output_path):
        return {
            **meta,
            "output_path": str(output_path),
            "processing_status": "skipped_existing",
            "error_message": "",
        }

    sequence_tensor = batch["sequence"][0]  # [T, C, H, W]
    frames = sequence_tensor[:, 0].detach().cpu().numpy().astype(np.float32)
    frame_indices = batch["frame_indices"][0].detach().cpu().numpy().astype(np.int64)
    frame_count = int(batch_scalar(batch, "frame_count")) if "frame_count" in batch else -1
    target_position = int(batch_scalar(batch, "target_idx")) if "target_idx" in batch else TARGET_POSITION
    stride = int(batch_scalar(batch, "temporal_stride")) if "temporal_stride" in batch else TEMPORAL_STRIDE

    unclamped = requested_unclamped_indices(meta["target_frame_index"])
    padding_mask = (unclamped != frame_indices)
    repeated_frame_count = int(SEQUENCE_LENGTH - len(np.unique(frame_indices)))

    assert frames.shape == (SEQUENCE_LENGTH, *IMAGE_SIZE)
    assert target_position == TARGET_POSITION
    assert frame_indices.shape == (SEQUENCE_LENGTH,)
    assert frame_indices[TARGET_POSITION] == meta["target_frame_index"]
    assert stride == TEMPORAL_STRIDE

    optical_flow, flow_magnitude, frame_difference = compute_motion_arrays(frames, flow_estimator)
    validate_motion_arrays(frames, optical_flow, flow_magnitude, frame_difference)

    all_zero_flow_transitions = int(np.sum(np.max(np.abs(optical_flow), axis=(1, 2, 3)) <= 1e-8))
    max_flow_magnitude = float(flow_magnitude.max())
    mean_flow_magnitude = float(flow_magnitude.mean())
    implausibly_large_flow = bool(max_flow_magnitude > QC_FLOW_WARNING_MAGNITUDE_PX)

    arrays = {
        "sample_id": np.asarray(sample_id, dtype="U128"),
        "video_id": np.asarray(meta["video_id"], dtype="U128"),
        "target_type": np.asarray(meta["target_type"], dtype="U16"),
        "target_frame_index": np.asarray(meta["target_frame_index"], dtype=np.int64),
        "frame_indices": frame_indices.astype(np.int64),
        "requested_unclamped_frame_indices": unclamped.astype(np.int64),
        "sequence_positions": np.arange(SEQUENCE_LENGTH, dtype=np.int64),
        "optical_flow": optical_flow.astype(np.float32),
        "flow_magnitude": flow_magnitude.astype(np.float32),
        "frame_difference": frame_difference.astype(np.float32),
        "stride": np.asarray(stride, dtype=np.int64),
        "target_position": np.asarray(target_position, dtype=np.int64),
        "padding_mask": padding_mask.astype(bool),
        "frame_count": np.asarray(frame_count, dtype=np.int64),
        "preprocessing_metadata": preprocessing_metadata_json(),
        "flow_method": np.asarray(flow_estimator.method_name, dtype="U64"),
        "flow_parameters": flow_config_json(),
        "flow_direction_convention": np.asarray(
            "optical_flow[t] maps sequence frame t toward sequence frame t+1; components are dx,dy in resized image pixels",
            dtype="U256",
        ),
    }
    if SAVE_INPUT_FRAMES:
        arrays["frames"] = frames.astype(np.float32)

    atomic_save_npz(output_path, **arrays)
    if not validate_saved_npz(output_path):
        raise RuntimeError(f"Saved file failed integrity check: {output_path}")

    return {
        **meta,
        "output_path": str(output_path),
        "first_selected_frame_index": int(frame_indices[0]),
        "last_selected_frame_index": int(frame_indices[-1]),
        "number_of_selected_frames": int(SEQUENCE_LENGTH),
        "number_of_valid_transitions": int(SEQUENCE_LENGTH - 1),
        "frame_height": int(frames.shape[1]),
        "frame_width": int(frames.shape[2]),
        "number_of_padded_or_clipped_frames": int(padding_mask.sum()),
        "number_of_repeated_frames": repeated_frame_count,
        "all_zero_flow_transitions": all_zero_flow_transitions,
        "max_flow_magnitude": max_flow_magnitude,
        "mean_flow_magnitude": mean_flow_magnitude,
        "implausibly_large_flow": implausibly_large_flow,
        "processing_status": "processed",
        "error_message": "",
    }


## Test-Set Processing Loop

The loop processes one sample at a time, catches failures, writes manifests incrementally, and supports safe resumption. In smoke mode it processes a few samples first.


In [ ]:
processed = skipped = failed = 0
for dataset_index, batch in enumerate(tqdm(motion_loader, desc=f"Motion extraction ({RUN_MODE})")):
    try:
        row = process_one_sample(dataset_index, batch)
        append_manifest_row(MANIFEST_PATH, row)
        if row["processing_status"] == "skipped_existing":
            skipped += 1
        else:
            processed += 1
    except Exception as exc:
        failed += 1
        try:
            meta = sample_meta(dataset_index, batch)
        except Exception:
            meta = {"sample_id": "unknown", "video_id": "unknown", "target_type": "unknown", "target_frame_index": -1}
        row = {
            **meta,
            "output_path": "",
            "first_selected_frame_index": None,
            "last_selected_frame_index": None,
            "number_of_selected_frames": 0,
            "number_of_valid_transitions": 0,
            "frame_height": None,
            "frame_width": None,
            "number_of_padded_or_clipped_frames": None,
            "number_of_repeated_frames": None,
            "all_zero_flow_transitions": None,
            "max_flow_magnitude": None,
            "mean_flow_magnitude": None,
            "implausibly_large_flow": None,
            "processing_status": "failed",
            "error_message": repr(exc),
        }
        append_manifest_row(MANIFEST_PATH, row)
        append_manifest_row(FAILED_PATH, row)
        print(f"Failed sample {dataset_index} / {row['sample_id']}: {exc}")
        if RUN_MODE == "smoke" and FAIL_FAST_SMOKE:
            raise
    finally:
        del batch
        gc.collect()

print(f"Done. processed={processed}, skipped={skipped}, failed={failed}")


## Quality-Control Sample Selection

When notebook 08 has a visualization manifest, this notebook prefers those sample IDs so motion QC corresponds to the same qualitative Grad-CAM examples. If that manifest is unavailable, it selects a deterministic subset from the motion manifest.


In [ ]:
def load_seg_gradcam_qc_ids() -> list[str]:
    candidates = [
        SEG_GRADCAM_OUTPUT_ROOT / "manifests" / "visualization_manifest.csv",
        SEG_GRADCAM_OUTPUT_ROOT / "visualization_manifest.csv",
    ]
    for path in candidates:
        if path.exists() and path.stat().st_size > 0:
            try:
                df = pd.read_csv(path)
            except pd.errors.EmptyDataError:
                continue
            if "sample_id" in df.columns:
                return [str(value) for value in df["sample_id"].dropna().tolist()]
    return []


def choose_qc_rows(manifest_df: pd.DataFrame) -> pd.DataFrame:
    manifest_df = manifest_df[manifest_df["processing_status"].isin(["processed", "skipped_existing"])].copy()
    if manifest_df.empty:
        return manifest_df
    preferred_ids = load_seg_gradcam_qc_ids()
    if preferred_ids:
        preferred = manifest_df[manifest_df["sample_id"].astype(str).isin(preferred_ids)]
        if not preferred.empty:
            return preferred.head(QC_SAMPLE_COUNT).reset_index(drop=True)
    return manifest_df.sample(n=min(QC_SAMPLE_COUNT, len(manifest_df)), random_state=QC_SEED).reset_index(drop=True)


manifest_df = read_manifest(MANIFEST_PATH)
qc_rows = choose_qc_rows(manifest_df)
qc_rows.to_csv(QC_MANIFEST_PATH, index=False)
print(f"QC samples selected: {len(qc_rows):,}")
display(qc_rows[["sample_id", "target_type", "output_path", "number_of_padded_or_clipped_frames", "max_flow_magnitude"]].head())


## Quality-Control Visualizations

These figures are only for inspection. They do not replace the numerical `.npz` arrays. Each transition panel states both source and destination original frame indices.


In [ ]:
def flow_to_hsv_rgb(flow: np.ndarray) -> np.ndarray:
    dx = flow[..., 0]
    dy = flow[..., 1]
    magnitude, angle = cv2.cartToPolar(dx, dy, angleInDegrees=False)
    hsv = np.zeros((*dx.shape, 3), dtype=np.uint8)
    hsv[..., 0] = np.mod(angle * 180 / np.pi / 2, 180).astype(np.uint8)
    hsv[..., 1] = 255
    norm_mag = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
    hsv[..., 2] = norm_mag.astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)


def overlay_flow_arrows(axis, flow: np.ndarray, step: int = QC_ARROW_STEP, color: str = "yellow") -> None:
    h, w = flow.shape[:2]
    y, x = np.mgrid[step // 2:h:step, step // 2:w:step]
    u = flow[y, x, 0]
    v = flow[y, x, 1]
    axis.quiver(x, y, u, v, color=color, angles="xy", scale_units="xy", scale=1.0, width=0.003)


def normalize_01(array: np.ndarray) -> np.ndarray:
    array = np.asarray(array, dtype=np.float32)
    finite = np.isfinite(array)
    if not finite.any():
        return np.zeros_like(array, dtype=np.float32)
    min_value = float(np.nanmin(array[finite]))
    max_value = float(np.nanmax(array[finite]))
    denom = max_value - min_value
    if denom <= 1e-8:
        return np.zeros_like(array, dtype=np.float32)
    normalized = (array - min_value) / denom
    return np.nan_to_num(normalized, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def heatmap_overlay_on_frame(
    frame: np.ndarray,
    heatmap: np.ndarray,
    colormap: int = cv2.COLORMAP_JET,
    alpha: float = 0.45,
) -> np.ndarray:
    frame_uint8 = np.clip(frame * 255.0, 0, 255).astype(np.uint8)
    frame_rgb = cv2.cvtColor(frame_uint8, cv2.COLOR_GRAY2RGB)
    heatmap_uint8 = np.clip(normalize_01(heatmap) * 255.0, 0, 255).astype(np.uint8)
    colored = cv2.cvtColor(cv2.applyColorMap(heatmap_uint8, colormap), cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(frame_rgb, 1.0 - alpha, colored, alpha, 0.0)


def write_motion_overlay_sheet(
    frames: np.ndarray,
    maps: np.ndarray,
    frame_indices: np.ndarray,
    selected_transitions: list[int],
    title_prefix: str,
    output_path: Path,
    colormap: int,
    alpha: float = 0.45,
) -> None:
    cols = min(4, len(selected_transitions))
    rows = int(math.ceil(len(selected_transitions) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.0), squeeze=False)
    for ax_idx, transition_idx in enumerate(selected_transitions):
        axis = axes.flat[ax_idx]
        overlay = heatmap_overlay_on_frame(
            frames[transition_idx],
            maps[transition_idx],
            colormap=colormap,
            alpha=alpha,
        )
        axis.imshow(overlay)
        axis.set_title(
            f"{title_prefix} overlay\nseq {transition_idx}->{transition_idx + 1}\nfrm {frame_indices[transition_idx]}->{frame_indices[transition_idx + 1]}",
            fontsize=8,
        )
        axis.axis("off")
    for axis in axes.flat[len(selected_transitions):]:
        axis.axis("off")
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def transition_indices_for_qc(total_transitions: int = SEQUENCE_LENGTH - 1) -> list[int]:
    if QC_MAX_TRANSITIONS_PER_SAMPLE >= total_transitions:
        return list(range(total_transitions))
    values = np.linspace(0, total_transitions - 1, QC_MAX_TRANSITIONS_PER_SAMPLE)
    return sorted({int(round(value)) for value in values})


def write_qc_visualization(row: pd.Series) -> dict[str, Any]:
    path = Path(row["output_path"])
    with np.load(path, allow_pickle=False) as data:
        frames = data["frames"].astype(np.float32) if "frames" in data else None
        frame_indices = data["frame_indices"].astype(int)
        optical_flow = data["optical_flow"].astype(np.float32)
        flow_magnitude = data["flow_magnitude"].astype(np.float32)
        frame_difference = data["frame_difference"].astype(np.float32)
    if frames is None:
        raise RuntimeError("QC visualization requires SAVE_INPUT_FRAMES=True outputs.")

    sample_dir = QC_DIR / sanitize_id(str(row["sample_id"]))
    sample_dir.mkdir(parents=True, exist_ok=True)

    # Input-frame contact sheet.
    cols = 5
    rows = int(math.ceil(SEQUENCE_LENGTH / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.2), squeeze=False)
    for idx, axis in enumerate(axes.flat):
        axis.axis("off")
        if idx < SEQUENCE_LENGTH:
            axis.imshow(frames[idx], cmap="gray", vmin=0, vmax=1)
            title = f"pos {idx}\nfrm {frame_indices[idx]}"
            if idx == TARGET_POSITION:
                title += "\ntarget"
                rect = plt.Rectangle((0, 0), frames.shape[2] - 1, frames.shape[1] - 1, fill=False, edgecolor="cyan", linewidth=3)
                axis.add_patch(rect)
            axis.set_title(title, fontsize=8)
    fig.tight_layout()
    fig.savefig(sample_dir / "selected_input_frames.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    selected_transitions = transition_indices_for_qc()
    for kind, arrays, cmap, filename in [
        ("Frame difference", frame_difference, "magma", "frame_difference_maps.png"),
        ("Flow magnitude", flow_magnitude, "viridis", "flow_magnitude_maps.png"),
    ]:
        cols = min(4, len(selected_transitions))
        rows = int(math.ceil(len(selected_transitions) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.0, rows * 2.8), squeeze=False)
        for ax_idx, transition_idx in enumerate(selected_transitions):
            axis = axes.flat[ax_idx]
            axis.imshow(arrays[transition_idx], cmap=cmap)
            axis.set_title(
                f"{kind}\nseq {transition_idx}->{transition_idx + 1}\nfrm {frame_indices[transition_idx]}->{frame_indices[transition_idx + 1]}",
                fontsize=8,
            )
            axis.axis("off")
        for axis in axes.flat[len(selected_transitions):]:
            axis.axis("off")
        fig.tight_layout()
        fig.savefig(sample_dir / filename, dpi=150, bbox_inches="tight")
        plt.close(fig)

    # Source-frame overlays for human QC. Numerical arrays remain the authoritative data.
    write_motion_overlay_sheet(
        frames,
        flow_magnitude,
        frame_indices,
        selected_transitions,
        "Flow magnitude",
        sample_dir / "source_frames_with_flow_magnitude_overlay.png",
        colormap=cv2.COLORMAP_JET,
        alpha=0.45,
    )
    write_motion_overlay_sheet(
        frames,
        frame_difference,
        frame_indices,
        selected_transitions,
        "Frame difference",
        sample_dir / "source_frames_with_frame_difference_overlay.png",
        colormap=cv2.COLORMAP_MAGMA,
        alpha=0.45,
    )

    # Direction/color and arrow overlays.
    cols = min(4, len(selected_transitions))
    rows = int(math.ceil(len(selected_transitions) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.0), squeeze=False)
    for ax_idx, transition_idx in enumerate(selected_transitions):
        axis = axes.flat[ax_idx]
        axis.imshow(flow_to_hsv_rgb(optical_flow[transition_idx]))
        axis.set_title(
            f"HSV flow\nseq {transition_idx}->{transition_idx + 1}\nfrm {frame_indices[transition_idx]}->{frame_indices[transition_idx + 1]}",
            fontsize=8,
        )
        axis.axis("off")
    for axis in axes.flat[len(selected_transitions):]:
        axis.axis("off")
    fig.tight_layout()
    fig.savefig(sample_dir / "flow_direction_hsv.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.0), squeeze=False)
    for ax_idx, transition_idx in enumerate(selected_transitions):
        axis = axes.flat[ax_idx]
        axis.imshow(frames[transition_idx], cmap="gray", vmin=0, vmax=1)
        overlay_flow_arrows(axis, optical_flow[transition_idx])
        axis.set_title(
            f"Arrows on source\nseq {transition_idx}->{transition_idx + 1}\nfrm {frame_indices[transition_idx]}->{frame_indices[transition_idx + 1]}",
            fontsize=8,
        )
        axis.axis("off")
    for axis in axes.flat[len(selected_transitions):]:
        axis.axis("off")
    fig.tight_layout()
    fig.savefig(sample_dir / "source_frames_with_flow_arrows.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    return {"sample_id": row["sample_id"], "qc_dir": str(sample_dir)}


qc_written = []
for _, row in tqdm(qc_rows.iterrows(), total=len(qc_rows), desc="Writing motion QC"):
    try:
        qc_written.append(write_qc_visualization(row))
    except Exception as exc:
        print(f"QC failed for {row.get('sample_id', 'unknown')}: {exc}")
    finally:
        plt.close("all")
        gc.collect()

qc_written_df = pd.DataFrame(qc_written)
qc_written_df.to_csv(QC_DIR / "written_qc_visualizations.csv", index=False)
print(f"QC visualization folders written: {len(qc_written_df):,}")


## Output Validation and Summary

The summary reports processed/skipped/failed counts, flow-quality warnings, repeated/clipped frame counts, and disk usage.


In [ ]:
def directory_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(item.stat().st_size for item in path.rglob("*") if item.is_file())


manifest_df = read_manifest(MANIFEST_PATH)
failed_df = read_manifest(FAILED_PATH)
valid_df = manifest_df[manifest_df["processing_status"].isin(["processed", "skipped_existing"])] if not manifest_df.empty else pd.DataFrame()

summary = {
    "run_mode": RUN_MODE,
    "active_samples": int(len(active_samples)),
    "manifest_rows": int(len(manifest_df)),
    "valid_rows": int(len(valid_df)),
    "failed_rows": int(len(failed_df)),
    "processed_rows": int((manifest_df["processing_status"] == "processed").sum()) if not manifest_df.empty else 0,
    "skipped_existing_rows": int((manifest_df["processing_status"] == "skipped_existing").sum()) if not manifest_df.empty else 0,
    "mean_flow_magnitude": float(valid_df["mean_flow_magnitude"].mean()) if not valid_df.empty else None,
    "max_flow_magnitude": float(valid_df["max_flow_magnitude"].max()) if not valid_df.empty else None,
    "implausibly_large_flow_samples": int(valid_df["implausibly_large_flow"].fillna(False).astype(bool).sum()) if not valid_df.empty else 0,
    "all_zero_flow_transition_total": int(valid_df["all_zero_flow_transitions"].fillna(0).sum()) if not valid_df.empty else 0,
    "padded_or_clipped_frame_total": int(valid_df["number_of_padded_or_clipped_frames"].fillna(0).sum()) if not valid_df.empty else 0,
    "repeated_frame_total": int(valid_df["number_of_repeated_frames"].fillna(0).sum()) if not valid_df.empty else 0,
    "output_disk_usage_bytes": directory_size_bytes(OUTPUT_ROOT),
    "qc_visualization_rows": int(len(qc_written_df)) if "qc_written_df" in globals() else 0,
    "flow_method": flow_estimator.method_name,
    "farneback_parameters": FLOW_CONFIG.__dict__,
    "flow_direction_convention": "optical_flow[t] maps sequence frame t toward sequence frame t+1; components are dx,dy in resized image pixels",
}

with SUMMARY_PATH.open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

print(json.dumps(summary, indent=2))
print(f"Summary saved to: {SUMMARY_PATH}")


## Loading Saved Motion Data

Example: load one saved `.npz` file and inspect shapes/dtypes.


In [ ]:
manifest_df = read_manifest(MANIFEST_PATH)
valid_df = manifest_df[manifest_df["processing_status"].isin(["processed", "skipped_existing"])] if not manifest_df.empty else pd.DataFrame()
if valid_df.empty:
    print("No valid saved motion samples yet.")
else:
    example_path = Path(valid_df.iloc[0]["output_path"])
    with np.load(example_path, allow_pickle=False) as data:
        print(f"Loaded: {example_path}")
        for key in data.files:
            array = data[key]
            print(f"{key}: shape={array.shape}, dtype={array.dtype}")


## Preparing Future Grad-CAM Warping

This notebook does not implement Grad-CAM warping, but this example shows how to load one optical-flow field, one Grad-CAM heatmap, and their associated frame indices.

The expected Grad-CAM `.npz` files from notebook 08 contain `encoder_cams`, `forward_cams`, `backward_cams_chronological`, and `frame_indices`. The `sample_id` from `motion_manifest.csv` can be joined with the Grad-CAM metrics/manifest by `sample_id`.


In [ ]:
def find_gradcam_npz_for_sample(sample_id: str) -> Path | None:
    arrays_dir = SEG_GRADCAM_OUTPUT_ROOT / "arrays"
    if not arrays_dir.exists():
        return None
    safe = sanitize_id(sample_id)
    matches = sorted(arrays_dir.glob(f"{safe}_*.npz"))
    return matches[0] if matches else None


if valid_df.empty:
    print("No motion outputs available for Grad-CAM loading example.")
else:
    row = valid_df.iloc[0]
    motion_path = Path(row["output_path"])
    gradcam_path = find_gradcam_npz_for_sample(str(row["sample_id"]))
    with np.load(motion_path, allow_pickle=False) as motion_data:
        transition_idx = 12
        flow = motion_data["optical_flow"][transition_idx]
        source_frame_index = int(motion_data["frame_indices"][transition_idx])
        destination_frame_index = int(motion_data["frame_indices"][transition_idx + 1])
        print(f"Motion sample: {row['sample_id']}")
        print(f"Flow transition {transition_idx}: frame {source_frame_index} -> {destination_frame_index}")
        print(f"Flow shape: {flow.shape}, dtype={flow.dtype}")
    if gradcam_path is None:
        print("No matching Grad-CAM npz found. Attach or point SEG_GRADCAM_OUTPUT_ROOT to notebook 08 outputs.")
    else:
        with np.load(gradcam_path, allow_pickle=False) as gradcam_data:
            heatmap = gradcam_data["encoder_cams"][transition_idx]
            print(f"Grad-CAM path: {gradcam_path}")
            print(f"Grad-CAM frame indices: {gradcam_data['frame_indices'][transition_idx]} -> {gradcam_data['frame_indices'][transition_idx + 1]}")
            print(f"Heatmap shape: {heatmap.shape}, dtype={heatmap.dtype}")


## Step-by-Step Kaggle Execution Instructions

1. Attach the project source-code dataset. Set `PROJECT_ROOT` and `FLAT_SOURCE_LAYOUT` in the configuration cell.
2. Attach raw EchoNet-Dynamic with `FileList.csv`, `VolumeTracings.csv`, and `Videos/`. Set `RAW_DIR`.
3. Attach processed masks with `metadata.csv`, `images/`, and `masks/`. Set `PROCESSED_DIR`.
4. If you want QC samples to match notebook 08, attach or preserve `/kaggle/working/seg_gradcam_outputs` and set `SEG_GRADCAM_OUTPUT_ROOT`.
5. Use any Kaggle accelerator setting. T4 x2 is fine, but Farneback optical flow runs on CPU and does not use multiple GPUs.
6. Start with `RUN_MODE = "smoke"` and run all cells through the processing loop and QC visualizations.
7. Confirm that sample `.npz` files contain shapes `[25, H, W]`, `[24, H, W, 2]`, `[24, H, W]`, and `[24, H, W]`.
8. Inspect QC figures for frame differences, flow magnitudes, HSV directions, flow arrows, duplicate boundary frames, and implausibly large motion.
9. Switch to `RUN_MODE = "full"`, set `FULL_MAX_SAMPLES = None`, and run the processing loop.
10. If interrupted, rerun with the same output directory and `OVERWRITE_EXISTING = False`; valid completed samples are skipped.
11. After full processing, run QC, summary, and loading-example cells.
12. Download or save `/kaggle/working/motion_metrics` as a Kaggle Dataset before ending the session.
